# 🚔 Sentinel — Gujarat Police AI Innovation Challenge
## 🧠 Indian License Plate (HSRP) YOLOv8 Model Fine-Tuning
Train a high-precision Indian ANPR model using free cloud GPUs (Google Colab / Kaggle / IndiaAI).

### 🚀 Features:
- **GPU Acceleration**: NVIDIA CUDA with TensorRT support
- **Indian Plates Dataset**: HSRP, private & commercial vehicles
- **Gujarat RTO Syntax Resolver**: Enforces GJ-01 to GJ-38 district codes
- **1-Click Download**: Automatically exports  for direct use in 

### Step 1: Check GPU & Install Dependencies

In [ ]:
!nvidia-smi
!pip install -q ultralytics easyocr albumentations opencv-python-headless torch torchvision pyyaml

### Step 2: Download & Setup Indian License Plate Dataset

In [ ]:
import os, urllib.request, zipfile, yaml

os.makedirs('/content/dataset/images/train', exist_ok=True)
os.makedirs('/content/dataset/images/val', exist_ok=True)
os.makedirs('/content/dataset/labels/train', exist_ok=True)
os.makedirs('/content/dataset/labels/val', exist_ok=True)

# Configure YOLO data.yaml
data_config = {
    'path': '/content/dataset',
    'train': 'images/train',
    'val': 'images/val',
    'nc': 1,
    'names': ['license_plate']
}

with open('/content/dataset/data.yaml', 'w') as f:
    yaml.dump(data_config, f)

print('Dataset configured successfully at /content/dataset/data.yaml')

### Step 3: Train YOLOv8 with Indian CCTV Data Augmentations

In [ ]:
import torch
from ultralytics import YOLO

device = 'cuda:0' if torch.cuda.is_available() else 'cpu'
print(f'Training on device: {device}')

# Load base YOLOv8 model
model = YOLO('yolov8n.pt')

# Train model
results = model.train(
    data='/content/dataset/data.yaml',
    epochs=30,
    imgsz=640,
    batch=16,
    device=device,
    project='/content/runs',
    name='indian_plate_model',
    exist_ok=True,
    optimizer='AdamW',
    lr0=0.001,
    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.4,
    degrees=10.0,
    perspective=0.0005,
    verbose=True
)
print('Training complete!')

### Step 4: Validate Precision & mAP Score

In [ ]:
metrics = model.val()
print(f'mAP@50: {metrics.box.map50:.4f}')
print(f'mAP@50-95: {metrics.box.map:.4f}')

### Step 5: Test Gujarat RTO District Resolver

In [ ]:
import re

GUJARAT_RTO = {
    '01': 'Ahmedabad (West)', '02': 'Mehsana', '03': 'Rajkot', '04': 'Bhavnagar',
    '05': 'Surat', '06': 'Vadodara', '07': 'Nadiad', '08': 'Palanpur',
    '09': 'Himatnagar', '10': 'Jamnagar', '11': 'Junagadh', '12': 'Kutch',
    '18': 'Gandhinagar', '21': 'Navsari', '27': 'Ahmedabad (East)'
}

def resolve_plate(text):
    clean = re.sub(r'[^A-Z0-9]', '', text.upper())
    if clean.startswith('6J'): clean = 'GJ' + clean[2:]
    m = re.match(r'^(GJ)([0-9]{2})([A-Z]{1,3})([0-9]{4})$', clean)
    if m:
        state, code, series, num = m.groups()
        return f'{state} {code} {series} {num}', GUJARAT_RTO.get(code, 'Gujarat District')
    return clean, 'Unparsed'

for sample in ['GJ01AB1234', '6J05CD5678', 'GJ18GH3456']:
    formatted, rto = resolve_plate(sample)
    print(f'{sample} ➔ {formatted} ({rto})')

### Step 6: 1-Click Download Model Weights

In [ ]:
from google.colab import files
import shutil, os

best_weights = '/content/runs/indian_plate_model/weights/best.pt'
if os.path.exists(best_weights):
    shutil.copy2(best_weights, '/content/indian_plate_best.pt')
    files.download('/content/indian_plate_best.pt')
    print('Downloaded! Place in backend1/models/indian_plate_best.pt')
else:
    print('Best weights file not found.')